# Análisis Exploratorio de Datos (EDA)

El objetivo de este análisis es conocer la estructura y calidad de la planilla de compras, describir sus principales variables cuantitativas y categóricas e identificar situaciones que puedan afectar los análisis posteriores.

El EDA constituye la etapa inicial de preparación de los datos antes de generar tablas de resumen y otros productos analíticos.

In [42]:
# 1. CONFIGURACIÓN, CARGA Y ESTRUCTURA GENERAL DEL DATASET

# Importar librerías
import pandas as pd
# import matplotlib.pyplot as plt

# ------------------------------------------------------------
# CARGA DEL ARCHIVO EXCEL
# ------------------------------------------------------------

df = pd.read_excel("plan_de_compras_2025.xlsx")

# ------------------------------------------------------------
# NORMALIZACIÓN DE LOS ENCABEZADOS
# ------------------------------------------------------------
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
)

print("Archivo cargado y encabezados normalizados correctamente.")

# ------------------------------------------------------------
# ESTRUCTURA GENERAL DEL DATASET
# ------------------------------------------------------------

estructura_df = pd.DataFrame({
    "N°": range(1, len(df.columns) + 1),
    "Nombre de la columna": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values
})

# ------------------------------------------------------------
# RESUMEN GENERAL
# ------------------------------------------------------------

print("DATASET CARGADO CORRECTAMENTE")
print("-" * 40)
print(f"Cantidad de filas:    {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")
print("-" * 40)

display(estructura_df)


Archivo cargado y encabezados normalizados correctamente.
DATASET CARGADO CORRECTAMENTE
----------------------------------------
Cantidad de filas:    831
Cantidad de columnas: 26
----------------------------------------


,N°,Nombre de la columna,Tipo de dato
0,1,unidad de compra,str
1,2,id proyecto,str
2,3,tipo proyecto,str
3,4,estado proyecto,str
4,5,código presupuestario,object
5,6,nombre proyecto,str
6,7,descripción proyecto,str
7,8,cantidad de ítems,int64
8,9,nombre ítem,str
9,10,tipo compra,int64


In [ ]:
# 2. CALIDAD Y CONTROL DE GRANULARIDAD DE LOS DATOS

calidad_df = pd.DataFrame({
    "Columna": df.columns,
    "Valores nulos": df.isnull().sum().values,
    "% Nulos": (df.isnull().mean().values * 100).round(2),
    "Valores únicos": df.nunique().values
})

duplicados = df.duplicated().sum()

print("CONTROL DE CALIDAD")
print("-" * 45)
print(f"Registros duplicados completos: {duplicados}")
print("-" * 45)

display(calidad_df)

# CONTROL DE GRANULARIDAD

df_analisis = df.copy()

columnas_sin_detalle_oc = [
    columna for columna in df_analisis.columns
    if columna not in [
        "cantidad oc asociadas ítem 2025",
        "oc asociada item 2025"
    ]
]

df_analisis = df_analisis.drop_duplicates(
    subset=columnas_sin_detalle_oc
)

print()
print("CONTROL DE GRANULARIDAD")
print("-" * 45)
print(f"Registros originales:        {len(df)}")
print(f"Registros para análisis:     {len(df_analisis)}")
print(
    f"Registros por detalle de OC: "
    f"{len(df) - len(df_analisis)}"
)
print("-" * 45)

CONTROL DE CALIDAD
---------------------------------------------
Registros duplicados completos: 0
---------------------------------------------


,Columna,Valores nulos,% Nulos,Valores únicos
0,unidad de compra,0,0.00,16
1,id proyecto,0,0.00,326
2,tipo proyecto,0,0.00,2
3,estado proyecto,0,0.00,2
4,código presupuestario,1,0.12,73
5,nombre proyecto,0,0.00,328
6,descripción proyecto,0,0.00,297
7,cantidad de ítems,0,0.00,3
8,nombre ítem,0,0.00,317
9,tipo compra,0,0.00,6



CONTROL DE GRANULARIDAD
---------------------------------------------
Registros originales:        831
Registros para análisis:     381
Registros por detalle de OC: 450
---------------------------------------------


In [ ]:
# 3. ESTADÍSTICAS DESCRIPTIVAS

columnas_numericas = [
    "cantidad de ítems",
    "cantidad productos",
    "monto unitario ítem",
    "monto total ítem año 2025",
    "cantidad oc",
    "cantidad oc asociadas ítem 2025",
    "monto de arrastre"
]

estadisticas_df = (
    df[columnas_numericas]
    .describe()
    .T
    .round(2)
)

estadisticas_df = estadisticas_df.rename(
    columns={
        "count": "Registros",
        "mean": "Promedio",
        "std": "Desv. estándar",
        "min": "Mínimo",
        "25%": "25%",
        "50%": "Mediana",
        "75%": "75%",
        "max": "Máximo"
    }
)

# Función formato CLP
def formato_clp(valor):
    return f"CLP $ {valor:,.0f}".replace(",", ".")

filas_montos = [
    "monto unitario ítem",
    "monto total ítem año 2025",
    "monto de arrastre"
]

columnas_estadisticas = [
    "Promedio",
    "Desv. estándar",
    "Mínimo",
    "25%",
    "Mediana",
    "75%",
    "Máximo"
]

idx = pd.IndexSlice

estadisticas_estilo = (
    estadisticas_df.style
    .format("{:.0f}", subset=idx[:, ["Registros"]])
    .format(
        formato_clp,
        subset=idx[filas_montos, columnas_estadisticas]
    )
    .set_properties(**{"text-align": "center"})
)

display(estadisticas_estilo)

,Registros,Promedio,Desv. estándar,Mínimo,25%,Mediana,75%,Máximo
cantidad de ítems,831,1.200000,0.410000,1.000000,1.000000,1.000000,1.000000,3.000000
cantidad productos,831,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
monto unitario ítem,831,CLP $ 108.521.418,CLP $ 203.621.987,CLP $ 90.000,CLP $ 4.200.000,CLP $ 40.000.000,CLP $ 202.000.000,CLP $ 4.839.000.000
monto total ítem año 2025,831,CLP $ 101.485.298,CLP $ 202.466.268,CLP $ 0,CLP $ 3.000.000,CLP $ 35.000.000,CLP $ 202.000.000,CLP $ 4.839.000.000
cantidad oc,831,1.340000,1.260000,1.000000,1.000000,1.000000,1.000000,12.000000
cantidad oc asociadas ítem 2025,831,125.760000,155.040000,0.000000,1.000000,4.000000,321.000000,321.000000
monto de arrastre,831,CLP $ 6.139.761,CLP $ 34.669.413,CLP $ 0,CLP $ 0,CLP $ 0,CLP $ 0,CLP $ 674.751.000


In [ ]:
# 4. RESUMEN DE VARIABLES CATEGÓRICAS

columnas_categoricas = [
    "unidad de compra",
    "tipo proyecto",
    "estado proyecto",
    "tipo compra"
]

resumen_categorias = []

for columna in columnas_categoricas:

    temp = (
        df[columna]
        .value_counts(dropna=False)
        .reset_index()
    )

    temp.columns = ["Categoría", "Cantidad"]

    temp["Participación (%)"] = (
        temp["Cantidad"] / len(df) * 100
    ).round(2)

    temp["Variable"] = columna

    resumen_categorias.append(temp)

categorias_df = pd.concat(
    resumen_categorias,
    ignore_index=True
)

categorias_mostrar = categorias_df.copy()

categorias_mostrar["Participación (%)"] = (
    categorias_mostrar["Participación (%)"]
    .map(lambda x: f"{x:.2f}%")
)

display(
    categorias_mostrar[
        [
            "Variable",
            "Categoría",
            "Cantidad",
            "Participación (%)"
        ]
    ]
)

,Variable,Categoría,Cantidad,Participación (%)
0,unidad de compra,Ministerio de Vivienda y Urbanismo(766),684,82.31%
1,unidad de compra,SEREMI MINVU V REGION,30,3.61%
2,unidad de compra,SEREMI MINVU III REGION,25,3.01%
3,unidad de compra,SEREMI MINVU VI REGION,20,2.41%
4,unidad de compra,SEREMI IV REGION,14,1.68%
5,unidad de compra,SEREMI MINVU REGION ARICA Y PARINACOTA,9,1.08%
6,unidad de compra,SEREMI MINVU XI REGION,8,0.96%
7,unidad de compra,SEREMI MINVU II REGION,8,0.96%
8,unidad de compra,SEREMI MINVU I REGION,7,0.84%
9,unidad de compra,SEREMI VIII REGION,6,0.72%


## Síntesis del Análisis Exploratorio de Datos

El análisis exploratorio permitió conocer la estructura, calidad y comportamiento general de la planilla de compras, compuesta por **831 registros y 26 variables**.

Se revisaron valores nulos, registros duplicados, valores únicos, variables cuantitativas y categóricas. Los montos fueron presentados en **pesos chilenos (CLP)** para facilitar su interpretación.

Un hallazgo relevante fue la existencia de registros asociados al detalle de órdenes de compra (OC). Al controlar esta granularidad se obtuvieron aproximadamente **381 registros para análisis**, identificándose **450 registros asociados al detalle de OC**. Esta consideración es importante para evitar sobreestimar montos en análisis posteriores.

La estadística descriptiva permitió identificar diferencias entre promedios, medianas y valores máximos, evidenciando la presencia de operaciones de alto valor. Asimismo, el análisis categórico permitió conocer la distribución y participación de las principales variables del dataset.

### Conclusión

El EDA permitió validar y comprender la información disponible, identificar aspectos relevantes de calidad y granularidad, y preparar una base analítica más confiable para la posterior construcción de **tablas de resumen y otros análisis**.